# 📊 Análise de Complexidade dos Jogos

Este notebook analisa a complexidade, fator de ramificação (branching factor) e tamanho do espaço de estados para diferentes jogos da nossa coleção.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'backend'))

from games.registry import get_game_class, GAMES
print("Módulos importados com sucesso!")

## 1. Algoritmo para Mapear Espaço de Estados

Usaremos uma busca em largura (BFS) a partir do estado inicial para contar o número de estados únicos alcançáveis e calcular o fator de ramificação médio.

In [ ]:
from collections import deque

def analyze_game(game_id, max_states=10000):
    game = get_game_class(game_id)
    initial_state = game.get_initial_state()
    
    visited = set()
    # Serialize state for tracking
    def get_state_key(s):
        return (tuple(s["board"]), s.get("current_player"), s.get("phase"), s.get("move_count", 0))
        
    queue = deque([initial_state])
    visited.add(get_state_key(initial_state))
    
    total_branches = 0
    evaluated_states = 0
    terminal_states = 0
    
    while queue and len(visited) < max_states:
        state = queue.popleft()
        evaluated_states += 1
        
        result = game.check_result(state)
        if result["over"]:
            terminal_states += 1
            continue
            
        moves = game.get_valid_moves(state)
        total_branches += len(moves)
        
        for move in moves:
            next_state = game.apply_move(game.clone_state(state), move)
            key = get_state_key(next_state)
            if key not in visited:
                visited.add(key)
                queue.append(next_state)
                
    avg_branching = total_branches / (evaluated_states - terminal_states) if (evaluated_states - terminal_states) > 0 else 0
    return {
        "states_analyzed": evaluated_states,
        "total_unique_visited": len(visited),
        "avg_branching_factor": avg_branching,
        "terminal_states": terminal_states,
        "incomplete": len(queue) > 0
    }

print("Função de análise pronta!")

## 2. Resultados da Análise de Complexidade

Vamos rodar a análise para os primeiros 6 jogos para comparar sua complexidade.

In [ ]:
target_games = [
    "tictactoe",
    "free-move",
    "tapatan",
    "shisima",
    "tsoro-yematatu",
    "mu-torere-1"
]

print(f"{'Jogo':<20} | {'Fator Ramif.':<12} | {'Estados Únicos':<15} | {'Terminais':<10}")
print("-" * 65)

for gid in target_games:
    game_info = next(g for g in GAMES if g["id"] == gid)
    analysis = analyze_game(gid)
    
    limit_suffix = "+" if analysis["incomplete"] else ""
    print(f"{game_info['name']:<20} | {analysis['avg_branching_factor']:<12.2f} | {analysis['total_unique_visited']:<15}{limit_suffix} | {analysis['terminal_states']:<10}")